# 通用單模型機率產生器（給 stacking 用）

換 `MODEL_NAME` / `TAG` 跑不同模型，各存一個 `probs_{TAG}.pkl`（含 OOF + test 機率）。
建議跑：roberta-wwm-ext、macbert-large、lert-base。macbert-base 你已有 probs_ensemble.pkl。

每個模型可單獨一個 session 跑（Save & Run All 自動關），跑完下載 pkl。


In [ ]:
# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 設定（每個模型改這裡）


In [ ]:
!pip install -q transformers==4.44.2 2>/dev/null

import os, json, glob, random, pickle, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

# ===== 每個模型改這三行 =====
MODEL_NAME = "hfl/chinese-roberta-wwm-ext"   # 換: hfl/chinese-macbert-large / hfl/chinese-lert-base
TAG        = "roberta"                        # 換: large / lert （決定 pkl 檔名）
BATCH_SIZE = 16                               # large 模型改 8
LR         = 2e-5                             # large 模型改 1e-5
# ============================
MAX_LEN=384; EPOCHS=8; CLASS_WEIGHT_MAX=5.0; USE_FGM=True; USE_AMP=True
SEEDS=[42]; N_SPLITS=5; OUT_DIR="/content/drive/MyDrive/AICUP"

EVAL_FIELDS = {"promise_status":["Yes","No"],
    "verification_timeline":["already","within_2_years","between_2_and_5_years","more_than_5_years","N/A"],
    "evidence_status":["Yes","No","N/A"],
    "evidence_quality":["Clear","Not Clear","Misleading","N/A"]}
FIELD_WEIGHTS={"promise_status":0.20,"verification_timeline":0.15,"evidence_status":0.30,"evidence_quality":0.35}
label2id={f:{l:i for i,l in enumerate(ls)} for f,ls in EVAL_FIELDS.items()}
num_labels={f:len(ls) for f,ls in EVAL_FIELDS.items()}
def set_seed(s): random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(MODEL_NAME, "| TAG:", TAG, "| device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.2 MB/s eta 0:00:00
hfl/chinese-roberta-wwm-ext | TAG: roberta | device: cuda


## 載入資料 + 元件


In [ ]:
def find_file(name):
    h=glob.glob(f"/content/drive/MyDrive/AICUP/{name}",recursive=True)
    if not h: raise FileNotFoundError(name)
    return h[0]
with open(find_file("vpesg4k_train_1000.json"),encoding="utf-8") as f: a=json.load(f)
with open(find_file("vpesg4k_val_1000.json"),  encoding="utf-8") as f: b=json.load(f)
train_data=a+b
with open(find_file("vpesg4k_test_2000.json"), encoding="utf-8") as f: test_data=json.load(f)
print(f"train {len(train_data)} | test {len(test_data)}")

class ESGDataset(Dataset):
    def __init__(s,data,tok,hl=True): s.data=data;s.tok=tok;s.hl=hl
    def __len__(s): return len(s.data)
    def __getitem__(s,i):
        x=s.data[i]; e=s.tok(x["data"],truncation=True,max_length=MAX_LEN,padding="max_length",return_tensors="pt")
        it={"input_ids":e["input_ids"].squeeze(0),"attention_mask":e["attention_mask"].squeeze(0)}
        if s.hl: it["labels"]={f:torch.tensor(label2id[f][x[f]],dtype=torch.long) for f in EVAL_FIELDS}
        return it

class MultiTaskModel(nn.Module):
    def __init__(s,mn,nl,dp=0.2):
        super().__init__(); s.bb=AutoModel.from_pretrained(mn); h=s.bb.config.hidden_size
        s.dp=nn.Dropout(dp); s.heads=nn.ModuleDict({f:nn.Linear(h,n) for f,n in nl.items()})
    def forward(s,ids,am):
        o=s.bb(input_ids=ids,attention_mask=am); last=o.last_hidden_state
        m=am.unsqueeze(-1).float(); pooled=(last*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return {f:hd(s.dp(pooled)) for f,hd in s.heads.items()}

class FGM:
    def __init__(s,m,eps=1.0): s.m=m;s.eps=eps;s.bak={}
    def attack(s,n="word_embeddings"):
        for nm,p in s.m.named_parameters():
            if p.requires_grad and n in nm and p.grad is not None:
                s.bak[nm]=p.data.clone(); nrm=torch.norm(p.grad)
                if nrm!=0 and not torch.isnan(nrm): p.data.add_(s.eps*p.grad/nrm)
    def restore(s,n="word_embeddings"):
        for nm,p in s.m.named_parameters():
            if p.requires_grad and n in nm and nm in s.bak: p.data=s.bak[nm]
        s.bak={}

def class_weights(data):
    cw={}
    for f,labs in EVAL_FIELDS.items():
        y=[label2id[f][s[f]] for s in data]; pr=np.unique(y); w=np.ones(len(labs),dtype=np.float32)
        for c,v in zip(pr,compute_class_weight("balanced",classes=pr,y=y)): w[c]=v
        cw[f]=torch.tensor(np.clip(w,None,CLASS_WEIGHT_MAX),dtype=torch.float32).to(device)
    return cw

def train_epoch(model,loader,opt,sch,cr,sc,fgm):
    model.train(); tot=0
    for b in loader:
        ids=b["input_ids"].to(device); am=b["attention_mask"].to(device)
        lb={f:b["labels"][f].to(device) for f in EVAL_FIELDS}
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            lo=model(ids,am); loss=sum(cr[f](lo[f],lb[f]) for f in EVAL_FIELDS)
        sc.scale(loss).backward()
        if fgm:
            fgm.attack()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                lo=model(ids,am); l2=sum(cr[f](lo[f],lb[f]) for f in EVAL_FIELDS)
            sc.scale(l2).backward(); fgm.restore()
        sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        sc.step(opt); sc.update(); sch.step(); tot+=loss.item()
    return tot/len(loader)

@torch.no_grad()
def predict(model,loader):
    model.eval(); pr={f:[] for f in EVAL_FIELDS}
    for b in loader:
        ids=b["input_ids"].to(device); am=b["attention_mask"].to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP): lo=model(ids,am)
        for f in EVAL_FIELDS: pr[f].append(torch.softmax(lo[f].float(),1).cpu().numpy())
    return {f:np.concatenate(pr[f],0) for f in EVAL_FIELDS}

def decode(probs):
    N=len(probs["promise_status"]); out=[]
    y,n=label2id["promise_status"]["Yes"],label2id["promise_status"]["No"]
    tl=["already","within_2_years","between_2_and_5_years","more_than_5_years"]; ti=[label2id["verification_timeline"][l] for l in tl]
    es=["Yes","No"]; ei=[label2id["evidence_status"][l] for l in es]
    eq=["Clear","Not Clear","Misleading"]; qi=[label2id["evidence_quality"][l] for l in eq]
    for i in range(N):
        ps="Yes" if probs["promise_status"][i,y]>=probs["promise_status"][i,n] else "No"
        r={"promise_status":ps}
        if ps=="No": r.update(verification_timeline="N/A",evidence_status="N/A",evidence_quality="N/A")
        else:
            r["verification_timeline"]=tl[int(np.argmax(probs["verification_timeline"][i,ti]))]
            e=es[int(np.argmax(probs["evidence_status"][i,ei]))]; r["evidence_status"]=e
            r["evidence_quality"]=eq[int(np.argmax(probs["evidence_quality"][i,qi]))] if e=="Yes" else "N/A"
        out.append(r)
    return out

def wmf1(gt,pred):
    s=0
    for f,labs in EVAL_FIELDS.items():
        s+=f1_score([g[f] for g in gt],[p[f] for p in pred],labels=labs,average="macro",zero_division=0)*FIELD_WEIGHTS[f]
    return s
print("元件就緒")

train 2000 | test 2000
元件就緒


## 訓練 + 存機率


In [ ]:
strat=[s["verification_timeline"] for s in train_data]
test_sum={f:np.zeros((len(test_data),num_labels[f])) for f in EVAL_FIELDS}
oof_sum={f:np.zeros((len(train_data),num_labels[f])) for f in EVAL_FIELDS}
tc=0
for sd in SEEDS:
    skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=sd); folds=list(skf.split(range(len(train_data)),strat))
    tok=AutoTokenizer.from_pretrained(MODEL_NAME)
    test_loader=DataLoader(ESGDataset(test_data,tok,False),batch_size=BATCH_SIZE,shuffle=False)
    oof={f:np.zeros((len(train_data),num_labels[f])) for f in EVAL_FIELDS}
    for fold,(tri,vai) in enumerate(folds):
        print(f"\nSEED {sd} Fold {fold+1}/{N_SPLITS}")
        tr=[train_data[i] for i in tri]; va=[train_data[i] for i in vai]
        trl=DataLoader(ESGDataset(tr,tok),batch_size=BATCH_SIZE,shuffle=True)
        val=DataLoader(ESGDataset(va,tok,False),batch_size=BATCH_SIZE,shuffle=False)
        set_seed(sd*100+fold)
        model=MultiTaskModel(MODEL_NAME,num_labels).to(device)
        cr={f:nn.CrossEntropyLoss(weight=class_weights(tr)[f]) for f in EVAL_FIELDS}
        opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
        ts=len(trl)*EPOCHS; sch=get_linear_schedule_with_warmup(opt,int(0.1*ts),ts)
        scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP); fgm=FGM(model) if USE_FGM else None
        best=-1; bp=None; bt=None
        for ep in range(EPOCHS):
            ls=train_epoch(model,trl,opt,sch,cr,scaler,fgm)
            vp=predict(model,val); s=wmf1(va,decode(vp))
            print(f"  epoch {ep+1}: loss={ls:.3f} val={s:.4f}")
            if s>best: best=s; bp=vp; bt=predict(model,test_loader)
        for f in EVAL_FIELDS: oof[f][vai]=bp[f]; test_sum[f]+=bt[f]
        tc+=1; del model; torch.cuda.empty_cache()
    for f in EVAL_FIELDS: oof_sum[f]+=oof[f]
oof_prob={f:oof_sum[f]/len(SEEDS) for f in EVAL_FIELDS}
test_prob={f:test_sum[f]/tc for f in EVAL_FIELDS}
print(f"\n{TAG} OOF CV = {wmf1(train_data,decode(oof_prob)):.4f}")
with open(f"{OUT_DIR}/probs_{TAG}.pkl","wb") as f:
    pickle.dump({"oof":oof_prob,"test":test_prob},f)
print(f"已存 probs_{TAG}.pkl")
from IPython.display import FileLink; FileLink(f"{OUT_DIR}/probs_{TAG}.pkl")

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


SEED 42 Fold 1/5


pytorch_model.bin:   0%|          | 0.00/412M [00:00<?, ?B/s]

  epoch 1: loss=4.389 val=0.5391
  epoch 2: loss=3.379 val=0.5315
  epoch 3: loss=2.573 val=0.6015
  epoch 4: loss=1.951 val=0.6085
  epoch 5: loss=1.475 val=0.6268
  epoch 6: loss=1.125 val=0.6140
  epoch 7: loss=0.885 val=0.6147
  epoch 8: loss=0.759 val=0.6238

SEED 42 Fold 2/5
  epoch 1: loss=4.365 val=0.4769
  epoch 2: loss=3.328 val=0.5252
  epoch 3: loss=2.622 val=0.5420
  epoch 4: loss=2.081 val=0.5910
  epoch 5: loss=1.615 val=0.5889
  epoch 6: loss=1.280 val=0.5881
  epoch 7: loss=1.053 val=0.6066
  epoch 8: loss=0.924 val=0.6039

SEED 42 Fold 3/5
  epoch 1: loss=4.394 val=0.4532
  epoch 2: loss=3.393 val=0.5789
  epoch 3: loss=2.556 val=0.5410
  epoch 4: loss=1.925 val=0.6023
  epoch 5: loss=1.459 val=0.6012
  epoch 6: loss=1.109 val=0.6122
  epoch 7: loss=0.882 val=0.6099
  epoch 8: loss=0.749 val=0.6190

SEED 42 Fold 4/5
  epoch 1: loss=4.406 val=0.5008
  epoch 2: loss=3.354 val=0.5006
  epoch 3: loss=2.619 val=0.5323
  epoch 4: loss=2.032 val=0.5799
  epoch 5: loss=1.535 

/content/drive/MyDrive/AICUP/probs_roberta.pkl